In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "jax"
os.environ["CBEAM_JAX_DEVICE_INDEX"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from multi_wvl_pipeline import *

"""
End-to-end example: propagate a batch of aberration configurations at
several native wavelengths, interpolate onto a fine grid, and hand the
result to the spectral extraction module.
"""

import specula
specula.init(0)
from specula.data_objects.ifunc import IFunc
from batch_propagation_pipeline import (
    get_simulation_parameters, create_random_aberration_configs,
    create_ramp_aberration_configs,
)
from spectral_extraction_module import SpectralImageSimulator, SpectralExtractor


import time
import numpy as np
 
from multi_wvl_pipeline import build_and_characterize_lantern_at_wavelength
from batch_propagation_pipeline import BatchPropagationPipeline
 

wvl_samples = 17
piston_samples = 81

minw = 760
maxw = 840

def profile_multiwavelength_preset(
    base_params, ifunc, native_wavelengths_nm,
    field_gen_on_gpu=False, field_gen_chunk_size=64,
):
    native_wavelengths_nm = np.asarray(sorted(native_wavelengths_nm), dtype=np.float64)
 
    char_times = []
    pipeline_times = []
    pupil_template = None
    shared_PL_N = build_lantern_geometry(base_params)
 
    for wl in native_wavelengths_nm:
        t0 = time.perf_counter()
        p_lambda, prop12 = build_and_characterize_lantern_at_wavelength(
            base_params, wl, PL_N=shared_PL_N, verbose=False)
        t1 = time.perf_counter()
 
        try:
            pipeline = BatchPropagationPipeline(
                prop12, p_lambda, ifunc,
                field_gen_on_gpu=field_gen_on_gpu,
                field_gen_chunk_size=field_gen_chunk_size,
                field_gen_pupil_template=pupil_template,
            )
        except TypeError:
            pipeline = BatchPropagationPipeline(
                prop12, p_lambda, ifunc,
                field_gen_on_gpu=field_gen_on_gpu,
                field_gen_chunk_size=field_gen_chunk_size,
            )
        t2 = time.perf_counter()
 
        if pupil_template is None:
            fg = pipeline._field_gen_np
            pupil_template = (fg.mask_np, fg.grid_size, fg.num_modes, fg.ifunc_matrix)
 
        char_times.append(t1 - t0)
        pipeline_times.append(t2 - t1)
        print(f"{wl:8.2f} nm   characterize/load: {t1-t0:7.3f} s   "
              f"pipeline build: {t2-t1:7.3f} s")
 
    char_times = np.array(char_times)
    pipeline_times = np.array(pipeline_times)
 
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"build_and_characterize_lantern_at_wavelength: "
          f"total {char_times.sum():7.2f} s   "
          f"mean {char_times.mean():6.3f} s   "
          f"min {char_times.min():6.3f} s   max {char_times.max():6.3f} s")
    print(f"BatchPropagationPipeline construction:         "
          f"total {pipeline_times.sum():7.2f} s   "
          f"mean {pipeline_times.mean():6.3f} s   "
          f"min {pipeline_times.min():6.3f} s   max {pipeline_times.max():6.3f} s")
    print(f"\nGrand total: {char_times.sum() + pipeline_times.sum():7.2f} s "
          f"over {len(native_wavelengths_nm)} wavelengths")
    print("=" * 60)
 
    return char_times, pipeline_times

print("=" * 60)
print("MULTI-WAVELENGTH PHOTONIC LANTERN PROPAGATION")
print("=" * 60)


In [ ]:
from cbeam.waveguide import hex_ring_positions
coords = hex_ring_positions(2, plot=True)
print(f"Number of positions: {len(coords)}")
coords = hex_ring_positions(3, plot=True)
print(f"Number of positions: {len(coords)}")
coords = hex_ring_positions(4, plot=True)
print(f"Number of positions: {len(coords)}")

In [ ]:
# =====================================================================
# EXPLORE: per-wavelength mode bookkeeping, 740..860 nm (step 10)
# =====================================================================
# Two untracked endpoint FEM solves at each wavelength -- NO
# characterize(), nothing written to disk. Shows where the degenerate-
# group / skipped-mode structure departs from the 800 nm hard-coded
# config, i.e. which cache tags are worth regenerating with
# auto_mode_bookkeeping=True.
#
# Cost: 2 endpoint FEM solves per wavelength; ~20 wavelengths is a
# few minutes -- widen EXPLORE_WL_NM for a denser scan.
# for a quicker first look.
# =====================================================================
from multi_wvl_pipeline import diagnose_mode_bookkeeping, build_lantern_geometry
from batch_propagation_pipeline import get_simulation_parameters

_bp = base_params if "base_params" in globals() else get_simulation_parameters(nrings=3)
_PL = build_lantern_geometry(_bp)          # shared geometry, built once

EXPLORE_WL_NM = list(range(minw, maxw+1, int( (maxw-minw) / (wvl_samples-1))) )

bookkeeping_by_wl = {}
for _wl in EXPLORE_WL_NM:
    print("\n" + "#" * 78 + f"\n#   {_wl} nm\n" + "#" * 78)
    _dgf, _dgb, _sk = diagnose_mode_bookkeeping(_bp, _wl, PL_N=_PL)
    bookkeeping_by_wl[_wl] = dict(degen_front=_dgf, degen_back=_dgb, skipped=list(_sk))

# ---- compact comparison against the 800 nm structure -----------------
_ref = bookkeeping_by_wl.get(800)
print("\n" + "=" * 78)
print("  SUMMARY   (flag => structure differs from 800 nm)")
print("=" * 78)
for _wl, _b in bookkeeping_by_wl.items():
    _flag = ""
    if _ref is not None and (
        _b["skipped"]     != _ref["skipped"]
        or _b["degen_front"] != _ref["degen_front"]
        or _b["degen_back"]  != _ref["degen_back"]
    ):
        _flag = "   <-- differs from 800 nm"
    print(f"  {_wl:>4} nm | skip={_b['skipped']} | "
          f"#front_groups={len(_b['degen_front'])} | "
          f"#back_groups={len(_b['degen_back'])}{_flag}")
    print(f"          front={_b['degen_front']}")

# Wavelengths whose structure differs from 800 nm are the ones to
# regenerate: delete their data/*/*_19port_XXXX_{front,back}.npy caches,
# then rebuild with MultiWavelengthPropagationPipeline(..., auto_mode_bookkeeping=True).


In [ ]:

# NOTE THIS! N_RINGS determines the number of output signals
base_params = get_simulation_parameters(nrings=3)
ifunc = IFunc.restore(base_params["ifunc_file"])

# Coarse native solve grid: physically propagate at these wavelengths.
native_wl_nm = np.linspace(minw, maxw, wvl_samples)

mwp = MultiWavelengthPropagationPipeline(
    base_params=base_params,
    ifunc=ifunc,
    native_wavelengths_nm=native_wl_nm,
    field_gen_on_gpu=False,
    field_gen_chunk_size=64,
    auto_mode_bookkeeping=True
)

n_total_modes = mwp.num_modes
coeff_matrix, labels = create_ramp_aberration_configs([0], piston_samples, -2000, 2000)

output_wl_nm = np.linspace(minw, maxw, wvl_samples)
power_spectra, wl_out = mwp.get_power_spectra(
    coeff_matrix,
    output_wavelengths_nm=output_wl_nm,
    use_gpu=False,
    prop_chunk_size=32,
)
print(f"\npower_spectra shape: {power_spectra.shape}  "
      f"(n_wavelengths={power_spectra.shape[0]}, "
      f"n_fields={power_spectra.shape[1]}, n_fibers={power_spectra.shape[2]})")


#  --- DIAGNOSTIC: run this once to see where preset-phase time goes ---
# char_times, pipeline_times = profile_multiwavelength_preset(
#     base_params, ifunc, native_wl_nm,
#     field_gen_on_gpu=False, field_gen_chunk_size=64,
# )
#  -----------------------------------------------------------------------


In [ ]:
spectral_cfg = build_spectral_config_from_wavelength_grid(wl_out)

In [ ]:
import numpy as np
import pickle

# --- After running the simulation ---
# Save with np.savez (or np.savez_compressed for large data)
np.savez('lantern19_results' + str(wvl_samples) + '.npz',
         power_spectra=power_spectra,
         wavelengths=wl_out,
         coeff_matrix=coeff_matrix,
         labels=labels)

# If you also need the spectral config for the extractor, you can save it
# with pickle (or rebuild it later from wl_out)
with open('spectral_cfg.pkl', 'wb') as f:
    pickle.dump(spectral_cfg, f)

# Also save fiber_positions if you need them for detector-stage plotting
# (they come from the lantern geometry, but you can recompute them from nrings)
# Or save them as well:
# np.savez('lantern_results.npz', ..., fiber_positions=fiber_positions)

print("Simulation results saved.")

In [ ]:
# NOTE: lantern19_results*.npz / spectral_cfg.pkl saved before the
# spectral_extraction_module -> spectral package split had extracted spectra
# sampled from the wrong detector columns (the extractor's wavelength<->pixel
# map didn't match the simulator's -- see spectral/dispersion.py). Re-run the
# simulate/save cells above to regenerate them if this file predates that fix.
import numpy as np
import pickle
from multi_wavelength_vis import *   # your plotting functions
from spectral_extraction_module import SpectralImageSimulator, SpectralExtractor
from multi_wvl_pipeline import build_spectral_config_from_wavelength_grid

with open('spectral_cfg.pkl', 'rb') as f:
    spectral_cfg = pickle.load(f)
# Load arrays
data = np.load('lantern19_results' + str(wvl_samples) + '.npz')
power_spectra = data['power_spectra']
wl_out = data['wavelengths']
coeff_matrix = data['coeff_matrix'] if 'coeff_matrix' in data else None
labels = data['labels'] if 'labels' in data else None

In [ ]:
simulator = SpectralImageSimulator(spectral_cfg)

In [ ]:
field_idx = int(piston_samples / 2)

In [ ]:
extractor = SpectralExtractor(spectral_cfg)

In [ ]:
spectra_field0 = spectra_for_field(power_spectra, field_idx=field_idx)

In [ ]:
image, fiber_positions = simulator.create_spectral_image(
    modal_powers=None, spectra=spectra_field0, add_noise=False, exposure_time=1.0,
)

In [ ]:
extracted, variance = extractor.extract_spectra(image, fiber_positions, method='aperture')
print(f"Extracted spectra shape: {extracted.shape}")

In [ ]:
# vedere come evolve la differenza fra psf non pistonata e psf pistonata per capire la scala spaziale
# fare plot parte immaginaria del piano focale

field_idx = int(piston_samples / 2)

from multi_wavelength_vis import *

# All spectra for one field, one per subplot
plot_spectra_grid(power_spectra, wl_out, field_idx=field_idx)
# A few fibers overlaid on one axes
plot_fiber_spectra(power_spectra, wl_out, field_idx=field_idx, fiber_indices=[0,1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], plot_sum=True)
plot_fiber_spectra(power_spectra, wl_out, field_idx=field_idx, fiber_indices=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,13,14,15,16,17,18], plot_sum=False)
# Wavelength x fiber heatmap for one field
plot_wavelength_fiber_heatmap(power_spectra, wl_out, field_idx=field_idx)
amplitudes = coeff_matrix[:, 0]  # the ramped mode's amplitude per field
plot_ramp_response(power_spectra, wl_out, amplitudes, fiber_idx=0, labels=labels)
plot_total_throughput_vs_amplitude(power_spectra, amplitudes, labels=labels, mode_label="mode 0")
plot_per_fiber_throughput_vs_amplitude(power_spectra, amplitudes, mode_label="mode 0")
# Detector-plane stage (image + extracted spectra) for one field, in one call
image, extracted, variance = plot_detector_stage_for_field(power_spectra, wl_out, field_idx=0)

In [ ]:
# Central output core (index 0) -- one curve per wavelength
import numpy as np
import matplotlib.pyplot as plt

if coeff_matrix is None:
    raise ValueError("coeff_matrix is missing; load lantern19_results33.npz with coeff_matrix saved.")

central_core_idx = 0
mode_idx = 0  # ramped mode in this notebook setup
amplitudes = np.asarray(coeff_matrix[:, mode_idx], dtype=float)
order = np.argsort(amplitudes)
amp_sorted = amplitudes[order]

fig, ax = plt.subplots(figsize=(10, 6))
cmap = plt.cm.viridis
norm = plt.Normalize(float(np.min(wl_out)), float(np.max(wl_out)))

for w_idx, wl in enumerate(wl_out):
    y = power_spectra[w_idx, order, central_core_idx]
    ax.plot(amp_sorted, y, color=cmap(norm(float(wl))), linewidth=1.0, alpha=0.95)

ax.set_xlabel("Aberration amplitude")
ax.set_ylabel("Central core power (a.u.)")
ax.set_title(f"Central output core (fiber {central_core_idx}) -- one curve per wavelength")
ax.grid(True, alpha=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Wavelength (nm)")

plt.tight_layout()
plt.show()

In [16]:
# =====================================================================
# CELL: Run diagnose_input_psf at every wavelength — NO propagation
# =====================================================================
# The PSF is evaluated at z=0, which is the WIDE (multimode) input end
# of the lantern.  At this end the waveguide is a single circular
# aperture of radius rclad — there are no individual cores yet.
# All intensity checks therefore use rclad as the relevant boundary.
# =====================================================================
import gc
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

from cbeam.propagator import Propagator
from batch_propagation_pipeline import IncidentFieldGenerator
from multi_wvl_pipeline import scale_params_to_wavelength, build_lantern_geometry

# ── Build lantern geometry + input mesh once ───────────────────────────────
shared_PL_N = build_lantern_geometry(base_params)

# solve_at(z=0) gives the mesh at the multimode input face — cheap, one-time.
# existing lines:
prop_ref    = Propagator(base_params["wl"], shared_PL_N,
                         base_params["n_output_positions"] + 1)
_, _        = prop_ref.solve_at(z=0)
mesh_pts_2d = prop_ref.mesh.points[:, :2]

# ADD THIS — same call already used in get_waveguide_properties:
from wavesolve.fe_solver import construct_B
B          = construct_B(prop_ref.mesh, sparse=True)
mesh_areas = np.array(B.diagonal())          # shape (N_pts,) — area of each FE node

rclad = base_params["rclad"]               # multimode input aperture radius
rjack = base_params["rjack"]               # outer jacket radius

# radial distance of every mesh node from the optical axis
r_mesh    = np.sqrt(mesh_pts_2d[:, 0]**2 + mesh_pts_2d[:, 1]**2)
in_clad   = r_mesh < rclad                 # inside the multimode core
in_jacket = (r_mesh >= rclad) & (r_mesh < rjack)   # cladding annulus

print(f"Input mesh: {len(mesh_pts_2d)} nodes  |  "
      f"inside rclad ({rclad} μm): {in_clad.sum()}  |  "
      f"in jacket annulus: {in_jacket.sum()}")

# ── Per-wavelength loop ─────────────────────────────────────────────────────
for wl_nm in native_wl_nm:

    print("\n" + "=" * 70)
    print(f"  PSF DIAGNOSTIC  –  {wl_nm:.1f} nm  (input multimode face, z=0)")
    print("=" * 70)

    # 1. Scale parameters to this wavelength
    p_lambda = scale_params_to_wavelength(base_params, wl_nm)

    # 2. Build (or reuse) the field generator
    if wl_nm == native_wl_nm[0]:
        fg = IncidentFieldGenerator(p_lambda, ifunc, xp=np)
        _pupil_template = (fg.mask_np, fg.grid_size, fg.num_modes, fg.ifunc_matrix)
    else:
        fg = IncidentFieldGenerator(p_lambda, ifunc, xp=np,
                                    pupil_template=_pupil_template)

    fg.precompute_interpolation_weights(mesh_pts_2d)

    # 3. Zero-aberration pupil → focal plane → mesh
    zero_coeffs   = np.zeros((1, fg.num_modes))
    Ef_input      = fg.generate_field_profiles_batch(zero_coeffs)      # (1, N, N)
    Ef_focal_grid = fg.apply_ef_to_lantern(Ef_input)[0]                # (Np, Np)
    Ef_focal_mesh = fg.resample_to_mesh(Ef_focal_grid[None, ...])[0]   # (Nmesh,)

    intensity_grid = np.abs(Ef_focal_grid) ** 2
    intensity_mesh = np.abs(Ef_focal_mesh) ** 2

    # ── INTENSITY CONSERVATION ────────────────────────────────────────────
    I_focal = intensity_grid.sum()
    I_mesh      = (intensity_mesh * mesh_areas).sum()
    frac_retained = I_mesh / I_focal if I_focal > 0 else 0.0

    print(f"\n[INTENSITY CONSERVATION]")
    print(f"  Total on FFT focal grid  : {I_focal:.6e}")
    print(f"  Total on FE mesh (all)   : {I_mesh:.6e}")
    print(f"  Fraction retained        : {frac_retained:.4f}  "
          f"({'OK' if frac_retained > 0.5 else '⚠️  LOW — check pixel_scale_um'})")

    # ── INTENSITY WITHIN THE MULTIMODE INPUT APERTURE ─────────────────────
    I_in_clad   = (intensity_mesh[in_clad]   * mesh_areas[in_clad]).sum()
    I_in_jacket = (intensity_mesh[in_jacket] * mesh_areas[in_jacket]).sum()
    outside_mask = ~(in_clad | in_jacket)
    I_outside   = (intensity_mesh[outside_mask] * mesh_areas[outside_mask]).sum()
    
    frac_clad   = I_in_clad   / I_focal if I_focal > 0 else 0.0
    frac_jacket = I_in_jacket / I_focal if I_focal > 0 else 0.0

    print(f"\n[INPUT APERTURE  rclad = {rclad:.1f} μm  rjack = {rjack:.1f} μm]")
    print(f"  Inside cladding  (r < {rclad:.1f} μm) : {I_in_clad:.6e}  "
          f"({frac_clad:.1%} of focal-grid total)  "
          f"{'✓' if frac_clad > 0.5 else '⚠️  LOW'}")
    print(f"  Jacket annulus                       : {I_in_jacket:.6e}  "
          f"({frac_jacket:.1%} of focal-grid total)")
    print(f"  Outside jacket                       : {I_outside:.6e}")
    if frac_clad < 0.5:
        print(f"  ⚠️  Less than half the PSF lands inside rclad — "
              f"beam may be too large or misaligned for this wavelength.")
    if frac_jacket > frac_clad:
        print(f"  ⚠️  More power in the jacket annulus than in the core — "
              f"likely a beam-size / pixel_scale_um mismatch.")

    # ── Mask centering ────────────────────────────────────────────────────
    mask            = fg.mask_np
    mask_center_y   = np.mean(np.where(mask > 0)[0])
    mask_center_x   = np.mean(np.where(mask > 0)[1])
    expected_center = (mask.shape[0] - 1) / 2.0

    print(f"\n[MASK CENTERING]")
    print(f"  Centroid ({mask_center_y:.2f}, {mask_center_x:.2f})  "
          f"Expected ({expected_center:.2f}, {expected_center:.2f})  "
          f"Offset ({mask_center_y-expected_center:.3f}, "
          f"{mask_center_x-expected_center:.3f}) px")
    if abs(mask_center_y-expected_center) > 0.1 or abs(mask_center_x-expected_center) > 0.1:
        print("  ⚠️  Mask is off-center!")

    # ── FFT grid centering ────────────────────────────────────────────────
    padded_size = fg.grid_size * p_lambda["pad_factor"]
    fft_center  = padded_size // 2
    peak_idx    = np.unravel_index(np.argmax(intensity_grid), intensity_grid.shape)
    offset_px   = (peak_idx[0] - fft_center, peak_idx[1] - fft_center)
    y_cen = np.sum(np.indices(intensity_grid.shape)[0] * intensity_grid) / I_focal
    x_cen = np.sum(np.indices(intensity_grid.shape)[1] * intensity_grid) / I_focal

    print(f"\n[FFT GRID]  pixel_scale = {p_lambda['pixel_scale_um']:.4f} μm/px")
    print(f"  Peak pixel {peak_idx}  offset {offset_px} px from center")
    print(f"  Centroid ({y_cen:.3f}, {x_cen:.3f})  "
          f"offset ({y_cen-fft_center:.3f}, {x_cen-fft_center:.3f}) px")
    if abs(offset_px[0]) > 0.5 or abs(offset_px[1]) > 0.5:
        print("  ⚠️  PSF peak off-center by > 0.5 px!")

    # ── PSF on mesh, annotated with cladding boundary ─────────────────────
    x_min, x_max = mesh_pts_2d[:, 0].min(), mesh_pts_2d[:, 0].max()
    y_min, y_max = mesh_pts_2d[:, 1].min(), mesh_pts_2d[:, 1].max()
    gx, gy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    mesh_grid_I = griddata((mesh_pts_2d[:, 0], mesh_pts_2d[:, 1]),
                            intensity_mesh, (gx, gy),
                            method='linear', fill_value=0)
    pk_mesh = np.unravel_index(np.argmax(mesh_grid_I), mesh_grid_I.shape)
    pk_mx   = gx[0, pk_mesh[1]]
    pk_my   = gy[pk_mesh[0], 0]
    dist_to_axis = np.sqrt(pk_mx**2 + pk_my**2)

    print(f"\n[MESH PSF PEAK]  ({pk_mx:.3f}, {pk_my:.3f}) μm  "
          f"dist from axis = {dist_to_axis:.3f} μm  "
          f"({'inside' if dist_to_axis < rclad else '⚠️ OUTSIDE'} cladding)")

    # ── 6-panel figure ────────────────────────────────────────────────────
    theta_circ = np.linspace(0, 2*np.pi, 300)   # for drawing circles

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(
        f"Input PSF — {wl_nm:.0f} nm  |  "
        f"retained: {frac_retained:.3f}  |  "
        f"in rclad: {frac_clad:.3f}",
        fontsize=12)

    # Panel 1: FFT focal plane (log)
    ax  = axes[0, 0]
    ext = [-fft_center, padded_size-fft_center,
           -fft_center, padded_size-fft_center]
    im  = ax.imshow(np.log10(intensity_grid + 1e-8), cmap='viridis',
                    extent=ext, origin='upper')
    ax.axvline(0, color='r', ls='--', alpha=0.5)
    ax.axhline(0, color='r', ls='--', alpha=0.5)
    ax.plot(offset_px[1], offset_px[0], 'r+', ms=15, mew=2, label='Peak')
    # draw rclad footprint in pixel units
    rclad_px = rclad / p_lambda["pixel_scale_um"]
    ax.plot(rclad_px * np.cos(theta_circ), rclad_px * np.sin(theta_circ),
            'c-', lw=1.5, label=f'rclad={rclad} μm')
    ax.set_title(f"FFT focal plane (log)  offset={offset_px}\n"
                 f"I_total={I_focal:.3e}")
    ax.set_xlabel("px"); ax.set_ylabel("px"); ax.legend(fontsize=7)
    plt.colorbar(im, ax=ax, label="log₁₀ I")

    # Panel 2: Pupil mask
    ax = axes[0, 1]
    im = ax.imshow(mask, cmap='gray', origin='lower')
    ax.axvline(expected_center, color='r', ls='--', alpha=0.5)
    ax.axhline(expected_center, color='r', ls='--', alpha=0.5)
    ax.plot(mask_center_x, mask_center_y, 'rx', ms=10, mew=2)
    ax.set_title("Pupil mask  (red × = centroid)")
    plt.colorbar(im, ax=ax)

    # Panel 3: PSF cross-sections
    ax      = axes[0, 2]
    x_axis  = np.arange(padded_size) - fft_center
    row_cut = intensity_grid[peak_idx[0], :]
    col_cut = intensity_grid[:, peak_idx[1]]
    ax.plot(x_axis, row_cut / row_cut.max(), label='Horizontal')
    ax.plot(x_axis, col_cut / col_cut.max(), label='Vertical')
    ax.axvline(0, color='r', ls='--', alpha=0.3)
    ax.axvline( rclad_px, color='c', ls=':', alpha=0.6, label=f'±rclad')
    ax.axvline(-rclad_px, color='c', ls=':', alpha=0.6)
    ax.set_title("Normalised PSF cross-sections")
    ax.set_xlabel("px from centre"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # Panel 4: PSF on input mesh with cladding + jacket circles
    ax = axes[1, 0]
    im = ax.imshow(mesh_grid_I, cmap='inferno',
                   extent=[x_min, x_max, y_min, y_max], origin='lower')
    ax.plot(rclad * np.cos(theta_circ), rclad * np.sin(theta_circ),
            'c-', lw=2, label=f'rclad={rclad} μm')
    ax.plot(rjack * np.cos(theta_circ), rjack * np.sin(theta_circ),
            'w--', lw=1.5, label=f'rjack={rjack} μm')
    ax.scatter(pk_mx, pk_my, c='red', s=80, marker='x', lw=2, label='PSF peak')
    ax.set_title(f"PSF on input mesh (z=0)\n"
                 f"I_mesh={I_mesh:.3e}  in rclad={frac_clad:.3f}")
    ax.set_xlabel("X (μm)"); ax.set_ylabel("Y (μm)")
    ax.legend(fontsize=7)
    plt.colorbar(im, ax=ax, label="Intensity")

    # Panel 5: Radial profile with cladding / jacket lines
    ax    = axes[1, 1]
    bins  = np.linspace(0, r_mesh.max(), 60)
    r_bin = (bins[1:] + bins[:-1]) / 2
    I_bin = np.array([
        np.average(intensity_mesh[sel], weights=mesh_areas[sel])
        if (sel := (r_mesh >= bins[i]) & (r_mesh < bins[i+1])).any() else 0.0
        for i in range(len(bins)-1)
    ])
    ax.plot(r_bin, I_bin, 'b-', lw=2)
    ax.axvline(rclad, color='c',  ls='--', lw=1.5, label=f'rclad={rclad} μm')
    ax.axvline(rjack, color='w' if False else 'gray',
               ls='--', lw=1.5, label=f'rjack={rjack} μm')
    ax.set_title("Radial PSF profile on mesh nodes")
    ax.set_xlabel("r from axis (μm)"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

    # Panel 6: Pupil phase
    ax = axes[1, 2]
    phase_input = np.angle(Ef_input[0])
    im = ax.imshow(phase_input, cmap='hsv', origin='lower',
                   vmin=-np.pi, vmax=np.pi)
    ax.set_title("Pupil phase (zero-aberration → flat)")
    plt.colorbar(im, ax=ax, label="Phase (rad)")

    plt.tight_layout()
    plt.show()

    del fg, Ef_input, Ef_focal_grid, Ef_focal_mesh, intensity_grid, intensity_mesh
    gc.collect()

print("\n✓  Per-wavelength PSF diagnostic complete.")